# SmolVLM: test-set mean visual attention (base vs reverse)
Enable Internet, select T4 ×2 and attach `huyhuy12334/cache-smolvlm`.
**Cell 1 clones/pulls the entire GitHub repository** and activates the updated source.
Push changes to the configured branch before rerunning cell 1.
Both base and reverse start at **batch 16/GPU**; only CUDA OOM reduces it.
tqdm shows cache validation, prompt preparation, each model and total measurement progress.

FP32. Mean heads then all 16,817 test questions. Final prompt query predicts the first answer token.
Token Y axes and spatial colors use log scales shared across both models.
Each model/plot type has its own cell; the last cell exports a ZIP.

New run: run cells in order. **Redraw saved results: cell 1, then cells 7–14 only.**
Outputs remain in `/kaggle/working/attention_outputs/`, outside the Git checkout.
Keep the original config/results when updating only figures; rerunning preparation after
source changes creates a new source fingerprint and is not needed for redraws.


In [ ]:
# 1. PULL ALL SOURCE from GitHub, install dependencies, and activate the updated code.
# Rerun this cell after pushing code; finish running workers before updating source.
import os
from pathlib import Path
SOURCE_REPOSITORY = "https://github.com/huyvanzzz/Log_Metrics.git"
SOURCE_BRANCH = "smolvlm-attention"  # Change only if your pushed code is on another branch.
SOURCE = Path('/kaggle/working/visualize-github-source')
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'

"""Standard-library-only Git setup, embedded verbatim in both Kaggle notebooks."""
import hashlib
import importlib
import importlib.util
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile


def ensure_no_workers(output_roots):
    for name in ('smolvlm_attention.runtime', 'smolvlm_attention.generated_runtime',
                 'dinov2_attention.runtime', 'smolvlm_probing.runtime',
                 'smolvlm_attention.sample_runtime', 'smolvlm_topk_benchmark.runtime'):
        module = sys.modules.get(name)
        if module is not None and any(process.poll() is None for process in getattr(module, '_PROCESSES', {}).values()):
            raise RuntimeError('A measurement worker is still running. Wait for it to finish before pulling source.')
    # The canonical probing output is shared by all notebooks. Also inspect it
    # after a kernel restart, when the new runtime is no longer in sys.modules.
    roots = set(map(Path, output_roots)) | {Path('/kaggle/working/smolvlm_probing_outputs'),
                                          Path('/kaggle/working/smolvlm_prompt_attention_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs'),
                                          Path('/kaggle/working/smolvlm_topk_benchmark_outputs_batch64')}
    for root in roots:
        for path in (Path(root) / 'logs').glob('*.pid.json'):
            pid = json.loads(path.read_text(encoding='utf-8'))['pid']
            try:
                os.kill(pid, 0)
            except (OSError, ProcessLookupError):
                continue
            raise RuntimeError(f'A measurement worker PID {pid} may still be running ({path}). Finish it before pulling.')


def sync_source(repository, branch, directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    def git(*args):
        return subprocess.check_output(['git', '-C', str(directory), *args], text=True).strip()
    if directory.exists():
        if not (directory / '.git').exists():
            raise RuntimeError(f'{directory} is not a Git checkout. Choose a new SOURCE directory.')
        if git('remote', 'get-url', 'origin').rstrip('/') != str(repository).rstrip('/'):
            raise RuntimeError('Existing source checkout has a different origin; choose a new SOURCE directory.')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Source checkout has local changes. Preserve them before pulling; no files were reset.')
        if git('branch', '--show-current') != branch:
            raise RuntimeError('Checkout branch differs from SOURCE_BRANCH. Use the matching branch or a new SOURCE directory.')
        subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', 'origin', branch], check=True)
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
                        str(repository), str(directory)], check=True)
    # Keep the historical source_archive_sha256 field reproducible without embedding stale source.
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for package in packages:
            folder = directory / package
            if not folder.is_dir():
                raise RuntimeError(f'Missing source package after pull: {package}')
            for path in sorted(folder.iterdir()):
                if path.suffix in ('.py', '.txt'):
                    entry = zipfile.ZipInfo(path.relative_to(directory).as_posix(), (2026, 1, 1, 0, 0, 0))
                    entry.compress_type = zipfile.ZIP_DEFLATED
                    archive.writestr(entry, path.read_bytes())
    return dict(commit=git('rev-parse', 'HEAD'), sha256=hashlib.sha256(buffer.getvalue()).hexdigest())


def activate_source(directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    for package in packages:
        for path in (directory / package).glob('*.py'):
            Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + '.')
               for package in packages):
            del sys.modules[name]
    sys.path[:] = [str(directory)] + [entry for entry in sys.path if entry != str(directory)]
    importlib.invalidate_caches()

ensure_no_workers([Path('/kaggle/working/attention_outputs'),
                   Path('/kaggle/working/generated_attention_outputs'),
                   Path('/kaggle/working/dinov2_attention_outputs')])
identity = sync_source(SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE)
SOURCE_COMMIT = identity["commit"]
SOURCE_ARCHIVE_SHA256 = identity["sha256"]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt')], check=True)
activate_source(SOURCE)
import torch, transformers, tokenizers
if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
    raise RuntimeError('Restart kernel after installing pinned packages, then run this setup cell again')
if torch.cuda.device_count() < 2:
    raise RuntimeError('Enable two GPUs, e.g. T4 x2')
OUTPUT = Path('/kaggle/working/attention_outputs')
RUN_CONFIG = OUTPUT / 'run.json'  # Existing results can be redrawn without rerunning preparation.
print({'source': str(SOURCE), 'commit': SOURCE_COMMIT, 'source_sha256': SOURCE_ARCHIVE_SHA256,
       'batch_per_gpu': 16, 'precision': 'float32',
       'gpus': [torch.cuda.get_device_name(i) for i in range(min(2, torch.cuda.device_count()))]})
print('Source updated. New run: continue cells in order. Redraw existing results: skip preparation/workers.')


In [ ]:
# 2. Pinned artifacts. Existing mounted cache is never copied into working.
from smolvlm_attention.runtime import resolve_artifacts
OUTPUT = Path('/kaggle/working/attention_outputs')
CACHE_INPUT_DIR = ''  # Optional: mounted directory containing the cache files.
artifacts = resolve_artifacts(OUTPUT, CACHE_INPUT_DIR)
print({name: spec for name, spec in artifacts['models'].items()})
print({'cache': artifacts['cache_dir'], 'prepared': artifacts['prepared_dir']})

In [ ]:
# 3. Full cache validation ONCE, then pretokenize test questions in both prompt orders.
from smolvlm_attention.runtime import prepare_run, launch_worker, wait_workers, load_results
RUN_CONFIG = prepare_run(artifacts, source_commit=SOURCE_COMMIT,
                         source_archive_sha256=SOURCE_ARCHIVE_SHA256)
print(RUN_CONFIG)

In [ ]:
# 4. Base: launch asynchronously on physical GPU 0.
base_worker = launch_worker(RUN_CONFIG, "base")

In [ ]:
# 5. Reverse: launch asynchronously on physical GPU 1.
reverse_worker = launch_worker(RUN_CONFIG, "reverse")

In [ ]:
# 6. tqdm for base, reverse and total; stage, processed count and batch are shown.
# Interrupting this wait leaves workers running; rerun to reconnect.
wait_workers(RUN_CONFIG)

In [ ]:
from smolvlm_attention.runtime import load_results
# 7. Require full test coverage for both models and compute SHARED plot scales.
means, model_metadata, scales, figure_metadata = load_results(RUN_CONFIG)
FIGURES = OUTPUT / 'figures'
from smolvlm_attention.plotting import (plot_token_atlas, plot_spatial_atlas,
                                       plot_layer_histogram, make_figures_zip)
from IPython.display import display, Image, FileLink
saved_run = json.loads(Path(RUN_CONFIG).read_text())
orders = {name: spec['camera_order'] for name, spec in saved_run['models'].items()}
print({name: {'count': model_metadata[name]['count'], 'shape': list(value.shape)}
       for name, value in means.items()})

In [ ]:
# BASE: 384 tokens, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_token_atlas(means["base"], orders["base"], FIGURES / "base",
    title='base: 384 tokens, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# BASE: six camera maps per layer, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_spatial_atlas(means["base"], orders["base"], FIGURES / "base",
    title='base: six camera maps per layer, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# BASE: distribution of 384 sample-averaged token values, all 30 layers. Re-run plotting without rerunning measurement.
paths = plot_layer_histogram(means["base"], orders["base"], FIGURES / "base",
    title='base: distribution of 384 sample-averaged token values, all 30 layers', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: 384 tokens, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_token_atlas(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: 384 tokens, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: six camera maps per layer, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_spatial_atlas(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: six camera maps per layer, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: distribution of 384 sample-averaged token values, all 30 layers. Re-run plotting without rerunning measurement.
paths = plot_layer_histogram(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: distribution of 384 sample-averaged token values, all 30 layers', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# 14. ZIP contains both figure folders (PNG/PDF) and measurement metadata only.
zip_path = make_figures_zip(FIGURES, OUTPUT / 'attention_figures.zip', figure_metadata)
print(f'{zip_path} | {zip_path.stat().st_size / 1024**2:.2f} MiB')
try:
    download_path = zip_path.relative_to(Path.cwd())
except ValueError:
    download_path = zip_path
display(FileLink(str(download_path)))

## Interpretation
The base best checkpoint is epoch 2; reverse is epoch 1. Compare their sample-averaged attention patterns, not a controlled camera-order intervention on identical weights. Histogram colors count averaged visual-token positions (384 per layer), not samples or heads. Camera heatmaps describe mean spatial positions, not any one source image.

If CUDA OOM occurs, workers reduce batch size without dropping records or switching precision. Progress snapshots allow matching runs to resume. A failed native parity check stops measurement; read `attention_outputs/logs/`. Outputs are complete only when both models cover every test ID.